# Stage1 日成交量参与度：冻结信息增量检验

完整 Stage1 原输入39项作为匹配对照，只新增一个“20日趋势为负时的成交量加权涨跌占比残差”。先核对 Qlib 来源、DatasetH 和标签成熟，不在结果出来后调整窗口、符号、模型或区间。2023+历史只作复用诊断，不能代表独立前向。

In [1]:
import torch
import json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd();card=root/'research/stage1_daily_volume_participation_information_20261022';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
source_spec=plan['candidate_source'];source_card=root/source_spec['card'];provider=root/source_spec['provider']
for file,key in [('qlib_store_manifest.json','provider_manifest_sha256')]:assert sha(provider/file)==source_spec[key]
for file,key in [('source_preflight_result.json','preflight_result_sha256'),('tail_bridge_result.json','tail_bridge_result_sha256')]:assert sha(source_card/file)==source_spec[key]
assert json.loads((source_card/'nas_source_archive_receipt.json').read_text())['archive_sha256']==source_spec['NAS_archive_sha256']
assert sha(root/plan['parent']['source_card']/'verification_manifest.json')==plan['parent']['verification_manifest_sha256']
parent_card=root/plan['parent']['source_card'];parent_manifest=json.loads((parent_card/'verification_manifest.json').read_text())
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:
    item=next(x for x in parent_manifest['entries'] if x['path'].endswith('/artifacts/'+name))
    assert item['sha256']==plan['parent'][key] and sha(parent_card/item['path'])==item['sha256']
qlib.init(provider_uri=str(provider),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time='2026-10-07',freq='day'))
price=D.features(['SOXX'],['$close','$volume'],start_time='2010-03-11',end_time='2026-10-07',freq='day').xs('SOXX',level='instrument').sort_index()
assert price.index.equals(calendar)
with R.uri_context(str(parent_card/'qlib_recorder')):
    recorder=R.get_recorder(recorder_id=plan['parent']['recorder_id'],experiment_id='905520356740140469')
    original=recorder.load_object('models/base_outer_71')
    base=recorder.load_object('inputs/features')
    frame=recorder.load_object('inputs/daily_frame')
    timing=recorder.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
ret=price['$close'].pct_change();sign=np.sign(ret);volume=price['$volume'].astype(float)
residual=((sign*volume).rolling(20,min_periods=20).sum()/volume.rolling(20,min_periods=20).sum()-sign.rolling(20,min_periods=20).mean()).replace([np.inf,-np.inf],np.nan)
feature=residual.reindex(frame.index).where(base.soxx_ret20.lt(0),0.0).rename('volume_participation_in_negative_trend')
forward=frame['soxx_forward_open_return'].astype(float)
parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert feature.loc[train_dates].notna().all() and feature.loc[held_dates].notna().all()
assert train_dates.sum()>=plan['cohorts']['minimum_mature_train_rows'] and held_dates.sum()>=plan['cohorts']['minimum_mature_held_rows']
augmented=base[base_names].copy();augmented[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
train_prepared=dataset.prepare('train',col_set=['feature','label','meta'])
held_prepared=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'source_and_DatasetH_maturity_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'parent_recorder_id':plan['parent']['recorder_id'],'source_provider_manifest_sha256':source_spec['provider_manifest_sha256'],'source_nas_archive_sha256':source_spec['NAS_archive_sha256'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'training_last_label_available_at':str(mature.loc[train_dates].max()),'held_last_label_available_at':str(mature.loc[held_dates].max()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_nonzero_rows':int(feature.loc[train_dates].ne(0).sum()),'candidate_held_nonzero_rows':int(feature.loc[held_dates].ne(0).sum()),'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False}
receipt_path=card/'preflight_result.json'
if receipt_path.exists():
    previous=json.loads(receipt_path.read_text())
    revision=json.loads((card/'plan_revision_before_fit.json').read_text())
    assert previous['plan_sha256']==revision['previous_plan_sha256'] and report['plan_sha256']==revision['current_plan_sha256']
    for field in ('training_mature_rows','held_mature_rows','original_primary_features','candidate_train_nonzero_rows','candidate_held_nonzero_rows'):
        assert previous[field]==report[field]
else:
    receipt_path.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print({k:report[k] for k in ['status','training_mature_rows','held_mature_rows','candidate_train_nonzero_rows','candidate_held_nonzero_rows']})

{'status': 'source_and_DatasetH_maturity_preflight_passed_no_fit', 'training_mature_rows': 3014, 'held_mature_rows': 940, 'candidate_train_nonzero_rows': 1132, 'candidate_held_nonzero_rows': 357}


## 固定两次拟合与一次配对信息门

先将两模型和评价预测封入同一个原生 Qlib Recorder；完整评价日历和两半区间均保留。执行前写入尝试计数，任何失败都不重置预算。

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
state_path=card/'run_state.json'
digest=sha(card/'plan.json')
revision=json.loads((card/'plan_revision_before_fit.json').read_text())
assert revision['current_plan_sha256']==digest and revision['model_fits_before_revision']==0
previous=json.loads((card/'preflight_result.json').read_text())
assert previous['plan_sha256']==revision['previous_plan_sha256']
state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'verified_source_object_loads':8,'fit_attempts':[],'fits_done':[],'bootstrap_attempted':False,'new_native_accounts':0}
assert state['plan_sha256']==digest and state['verified_source_object_loads']==plan['budget']['verified_source_object_loads_max']
assert not state['fit_attempts'] and not state['bootstrap_attempted']
tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)]
he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows'] and len(he)==report['held_mature_rows']
yname='soxx_forward_five_open_log_return';name='volume_participation_in_negative_trend'
ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
xb=tr['feature'][base_names].to_numpy(float);hb=he['feature'][base_names].to_numpy(float)
xc=tr['feature'][base_names+[name]].to_numpy(float);hc=he['feature'][base_names+[name]].to_numpy(float)
assert np.isfinite(xc[:,-1]).all() and np.isfinite(hc[:,-1]).all()
tracking=card/'qlib_recorder'
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    rec=R.get_recorder()
    state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id))
    state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
    rec.save_objects(artifact_path='inputs',plan=plan,preflight=previous,revision=revision,dataset=dataset)
    for family,xtrain,xheld in [('baseline',xb,hb),('candidate',xc,hc)]:
        assert len(state['fit_attempts'])<plan['budget']['model_fits_max'] and family not in state['fit_attempts']
        state['fit_attempts'].append(family)
        state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
        model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
        model.fit(xtrain,ytr)
        prediction=model.predict(xheld)
        assert np.isfinite(prediction).all()
        rec.save_objects(artifact_path='models',**{family:model})
        rec.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})})
        state['fits_done'].append(family)
        state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
        rec.log_metrics(completed_fits=len(state['fits_done']))
        models[family],predictions[family]=model,prediction
    assert len(state['fits_done'])==2
    dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
    daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
    daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2
    daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2
    daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
    assert daily.signal_date.is_monotonic_increasing and len(daily)==report['held_mature_rows']
    boundary=pd.Timestamp('2025-01-01')
    halves=[daily.loc[daily.signal_date.lt(boundary)],daily.loc[daily.signal_date.ge(boundary)]]
    assert min(map(len,halves))>300
    alpha=plan['information_gate']['paired_stationary_bootstrap']['one_sided_alpha']
    b=plan['information_gate']['paired_stationary_bootstrap']
    state['bootstrap_attempted']=True
    state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
    interval,_,replicates=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),mean_block=b['mean_block_sessions'],resamples=b['draws'],seed=b['seed'],alpha=alpha)
    coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
    metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),'candidate_held_mse':float(daily.candidate_squared_error.mean()),'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],'candidate_standardized_coordinate_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
    passed=bool(coefficient>0 and metrics['held_paired_mse_improvement']>0 and all(x>0 for x in metrics['held_half_paired_improvements']) and metrics['bootstrap_lower']>0)
    rec.save_objects(artifact_path='analysis/daily',held=daily)
    rec.save_objects(artifact_path='analysis',metrics=metrics,bootstrap_replicates=replicates,information_gate_pass=passed)
    rec.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'plan_sha256':digest,'decision':'information_pass_historical_only_needs_0bp_design' if passed else 'not_promoted_fixed_daily_volume_information_failed','objective_achieved':False,'information_gate_pass':passed,'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'reused_market_history':True,'historical_first_receipt_certified':False,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/state['experiment_id']/state['recorder_id']),'artifact_policy':plan['artifact_policy']}
(card/'result_review.json').write_text(json.dumps({'status':summary['decision'],'plan_sha256':digest,'training_mature_rows':len(tr),'held_mature_rows':len(he),**metrics,'information_gate_pass':passed,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'historical_independence':False,'historical_first_receipt_certified':False,'recorder_id':state['recorder_id']},ensure_ascii=False,indent=2)+'\n')
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
state['completed']=True
state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
print({k:metrics[k] for k in ('baseline_held_mse','candidate_held_mse','held_paired_mse_improvement','held_half_paired_improvements','candidate_standardized_coordinate_coefficient','bootstrap_lower')},'gate',passed)


## 中文结论

本预检只核对来源与标签成熟；计划固定的两次 Ridge 拟合和一次区块检验必须按尝试记录执行。若信息门失败，关闭固定特征；若通过，也仅能另冻结 Stage1 0bp 全账户实验。